# Smart Sales Forecasting System — Phase 1: ML/EDA

**Goal:** understand the forecasting data, validate it, identify temporal patterns, define targets/features, and prepare chronological train/validation/test datasets.

> **Data provenance:** the original uploaded catalog is reused, while Quantity, Promotion, Discount, Holiday, Inventory, Lead Time and Reorder Point are synthetic variables generated for this project. They must be labeled synthetic in the final documentation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('/mnt/data/smart_sales_forecasting_dataset')
DAILY = DATA_DIR / 'synthetic_daily_forecasting.csv'
PRODUCT_DAILY = DATA_DIR / 'synthetic_product_daily_forecasting.csv'

daily = pd.read_csv(DAILY, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
product_daily = pd.read_csv(PRODUCT_DAILY, parse_dates=['Date']).sort_values(['Product_ID','Date']).reset_index(drop=True)
print('Daily shape:', daily.shape)
print('Product-day shape:', product_daily.shape)


## 1. Data quality checks

In [ ]:
print('Date range:', daily.Date.min().date(), 'to', daily.Date.max().date())
print('Unique dates:', daily.Date.nunique())
print('Expected calendar days:', (daily.Date.max()-daily.Date.min()).days + 1)
print('Missing values in daily data:')
display(daily.isna().sum().to_frame('missing').query('missing > 0'))
print('Duplicate daily dates:', daily.duplicated('Date').sum())
print('Negative quantity:', (daily.Quantity < 0).sum())
print('Negative sales:', (daily.Sales_Amount < 0).sum())
print('Products:', product_daily.Product_ID.nunique())
print('Duplicate product-date keys:', product_daily.duplicated(['Product_ID','Date']).sum())
print('Negative product quantity:', (product_daily.Quantity < 0).sum())


## 2. Descriptive statistics

In [ ]:
display(daily[['Quantity','Sales_Amount','Profit','Products_Sold']].describe().T)
product_summary=(product_daily.groupby('Product_ID').agg(total_quantity=('Quantity','sum'),total_sales=('Sales_Amount','sum'),avg_daily_quantity=('Quantity','mean'),avg_daily_sales=('Sales_Amount','mean')).sort_values('total_sales',ascending=False))
display(product_summary.head(20))


## 3. Overall trend

In [ ]:
fig, ax=plt.subplots(figsize=(14,5))
ax.plot(daily.Date,daily.Sales_Amount)
ax.set_title('Daily Sales Revenue')
ax.set_xlabel('Date'); ax.set_ylabel('Sales Amount')
plt.tight_layout(); plt.show()

fig, ax=plt.subplots(figsize=(14,5))
ax.plot(daily.Date,daily.Quantity)
ax.set_title('Daily Demand / Quantity')
ax.set_xlabel('Date'); ax.set_ylabel('Quantity')
plt.tight_layout(); plt.show()


## 4. Weekly seasonality

In [ ]:
daily['Day_Name']=daily.Date.dt.day_name()
weekly=daily.groupby('Day_Name').agg(avg_quantity=('Quantity','mean'),avg_sales=('Sales_Amount','mean')).reindex(['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
display(weekly)
fig,ax=plt.subplots(figsize=(10,5)); weekly.avg_quantity.plot(kind='bar',ax=ax); ax.set_title('Average Demand by Day of Week'); ax.set_ylabel('Average Quantity'); plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## 5. Monthly and yearly patterns

In [ ]:
daily['Year']=daily.Date.dt.year
daily['Month']=daily.Date.dt.month
monthly=daily.groupby('Month').agg(avg_quantity=('Quantity','mean'),avg_sales=('Sales_Amount','mean'))
annual=daily.groupby('Year').agg(total_quantity=('Quantity','sum'),total_sales=('Sales_Amount','sum'),total_profit=('Profit','sum'))
display(monthly); display(annual)
fig,ax=plt.subplots(figsize=(10,5)); monthly.avg_quantity.plot(kind='bar',ax=ax); ax.set_title('Average Demand by Month'); ax.set_ylabel('Average Quantity'); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(12,5)); annual.total_sales.plot(marker='o',ax=ax); ax.set_title('Annual Sales'); ax.set_ylabel('Sales Amount'); plt.tight_layout(); plt.show()


## 6. Promotion and holiday effects

In [ ]:
# Promotions is a daily count in the aggregate file; treat >0 as promotion day.
daily['Promotion_Day']=(daily.Promotions>0).astype(int)
promo=daily.groupby('Promotion_Day').agg(avg_quantity=('Quantity','mean'),avg_sales=('Sales_Amount','mean'),avg_profit=('Profit','mean'))
holiday=daily.groupby('Holiday_Flag').agg(avg_quantity=('Quantity','mean'),avg_sales=('Sales_Amount','mean'),avg_profit=('Profit','mean'))
display(promo); display(holiday)


## 7. Product/category analysis

In [ ]:
category_summary=(product_daily.groupby('Category_ID').agg(total_quantity=('Quantity','sum'),total_sales=('Sales_Amount','sum'),total_profit=('Profit','sum')).sort_values('total_sales',ascending=False))
display(category_summary)


## 8. Inventory risk

In [ ]:
x=product_daily.copy()
x['Reorder_Risk']=x.Current_Inventory <= x.Reorder_Point
x['Stock_Cover_Days']=x.Current_Inventory / x.Quantity.replace(0,np.nan)
print('Share of product-days at/below reorder point:', round(x.Reorder_Risk.mean()*100,2),'%')
display(x[['Current_Inventory','Reorder_Point','Supplier_Lead_Time_Days','Stock_Cover_Days']].describe().T)


## 9. Autocorrelation

In [ ]:
series=daily.set_index('Date').Quantity.astype(float)
lags=[1,7,14,28,30,365]
acf={lag:series.autocorr(lag=lag) for lag in lags}
display(pd.Series(acf,name='autocorrelation').to_frame())
fig,ax=plt.subplots(figsize=(12,5)); pd.Series({lag:series.autocorr(lag=lag) for lag in range(1,61)}).plot(kind='bar',ax=ax); ax.set_title('Demand Autocorrelation, Lags 1–60'); ax.set_xlabel('Lag'); ax.set_ylabel('Autocorrelation'); plt.tight_layout(); plt.show()


## 10. Supervised-learning features

In [ ]:
model_daily=daily[['Date','Quantity','Sales_Amount','Profit','Promotions','Holiday_Flag']].copy()
model_daily['day_of_week']=model_daily.Date.dt.dayofweek
model_daily['day_of_month']=model_daily.Date.dt.day
model_daily['week_of_year']=model_daily.Date.dt.isocalendar().week.astype(int)
model_daily['month']=model_daily.Date.dt.month
model_daily['quarter']=model_daily.Date.dt.quarter
model_daily['is_weekend']=(model_daily.day_of_week>=5).astype(int)
model_daily['days_since_start']=(model_daily.Date-model_daily.Date.min()).dt.days
for lag in [1,7,14,28]:
    model_daily[f'quantity_lag_{lag}']=model_daily.Quantity.shift(lag)
    model_daily[f'sales_lag_{lag}']=model_daily.Sales_Amount.shift(lag)
for w in [7,14,28]:
    model_daily[f'quantity_rolling_mean_{w}']=model_daily.Quantity.shift(1).rolling(w).mean()
    model_daily[f'quantity_rolling_std_{w}']=model_daily.Quantity.shift(1).rolling(w).std()
    model_daily[f'sales_rolling_mean_{w}']=model_daily.Sales_Amount.shift(1).rolling(w).mean()
model_daily=model_daily.dropna().reset_index(drop=True)
display(model_daily.head())


## 11. Chronological train / validation / test split

In [ ]:
train=model_daily[model_daily.Date<'2024-01-01'].copy()
validation=model_daily[(model_daily.Date>='2024-01-01')&(model_daily.Date<'2025-01-01')].copy()
test=model_daily[model_daily.Date>='2025-01-01'].copy()
print('TRAIN:',train.Date.min().date(),'→',train.Date.max().date(),len(train))
print('VALIDATION:',validation.Date.min().date(),'→',validation.Date.max().date(),len(validation))
print('TEST:',test.Date.min().date(),'→',test.Date.max().date(),len(test))


## 12. Phase 1 decision

**Primary forecasting target:** daily `Quantity` (demand).

**Secondary forecasting target:** daily `Sales_Amount` (revenue).

**Horizons:** 7, 30 and 90 days.

**Candidate models:** Naive baseline → Seasonal Naive → Scikit-learn model(s) → TensorFlow LSTM.

**Evaluation:** MAE, RMSE, MAPE and WAPE on chronological validation/test data. No random shuffling.

The final production model will be selected by measured out-of-sample performance, not by assuming LSTM is best.

In [ ]:
processed_path=DATA_DIR/'processed_daily_forecasting_features.csv'
model_daily.to_csv(processed_path,index=False)
print('Saved:',processed_path)
